# MailSense — training on Kaggle GPU

Runs the same scripts as the local repository; only the paths change. Nothing
Kaggle-specific lives inside `src/` — the dataset, results and model directories
are all passed in as arguments.

**Setup:** add the repository (or just `src/`, `configs/` and `data/`) as a
Kaggle dataset, then set `REPO` below. Turn the GPU accelerator on for BERT.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPO = Path('/kaggle/input/mailsense')      # <- folder holding src/, configs/, data/
WORK = Path('/kaggle/working')
DATASET = REPO / 'data' / 'Ask0729-fixed.txt'
SPLITS = WORK / 'data' / 'splits'
RESULTS = WORK / 'results'
MODELS = WORK / 'models'

# Kaggle input is read-only, so copy the code into the writable working dir.
if REPO.exists():
    subprocess.run(['cp', '-r', str(REPO / 'src'), str(WORK)], check=True)
    subprocess.run(['cp', '-r', str(REPO / 'configs'), str(WORK)], check=True)
os.chdir(WORK)
sys.path.insert(0, str(WORK))

import torch
print('torch', torch.__version__, '| cuda available:', torch.cuda.is_available())

## 1. Build the splits (once — all three models reuse them)

In [ ]:
def run(cmd):
    print('$', ' '.join(str(c) for c in cmd))
    subprocess.run([sys.executable, *[str(c) for c in cmd]], check=True)

run(['-m', 'src.preprocessing.prepare_data',
     '--dataset', DATASET,
     '--output-dir', WORK / 'data',
     '--results-dir', RESULTS,
     '--seed', 42])

## 2. Model A — TF-IDF + Logistic Regression (CPU, seconds)

In [ ]:
run(['-m', 'src.tfidf.train_tfidf',
     '--config', WORK / 'configs' / 'tfidf.json',
     '--splits-dir', SPLITS, '--results-dir', RESULTS, '--models-dir', MODELS,
     '--seed', 42])

## 3. Model B — LSTM (GPU, a few minutes)

In [ ]:
run(['-m', 'src.lstm.train_lstm',
     '--config', WORK / 'configs' / 'lstm.json',
     '--splits-dir', SPLITS, '--results-dir', RESULTS, '--models-dir', MODELS,
     '--epochs', 20, '--batch-size', 32, '--learning-rate', 1e-3,
     '--max-len', 64, '--seed', 42, '--device', 'auto'])

## 4. Model C — BERT (GPU)

Standard `bert-base-uncased` fine-tuning: nothing frozen, nothing removed. If the
GPU runs out of memory, reduce `--batch-size` and raise
`--gradient-accumulation-steps` so the effective batch size stays the same, and
record the change.

In [ ]:
run(['-m', 'src.bert.train_bert',
     '--config', WORK / 'configs' / 'bert.json',
     '--splits-dir', SPLITS, '--results-dir', RESULTS, '--models-dir', MODELS,
     '--pretrained-model', 'bert-base-uncased',
     '--epochs', 4, '--batch-size', 16, '--learning-rate', 2e-5,
     '--max-len', 64, '--seed', 42, '--device', 'auto'])

## 5. Comparison table + confusion-matrix figures

In [ ]:
run(['-m', 'src.evaluation.compare', '--results-dir', RESULTS])
print((RESULTS / 'comparison_test.md').read_text())

## 6. Collect the artefacts

Download `/kaggle/working/results` and commit it back to the repository: metrics
JSON, confusion matrices, training histories and the comparison table are all
small, Git-friendly text files. Leave the model checkpoints out of Git.

In [ ]:
for p in sorted(RESULTS.rglob('*')):
    if p.is_file():
        print(f'{p.stat().st_size:>9,}  {p.relative_to(RESULTS)}')